<a href="https://colab.research.google.com/github/TediBalint/AI-Jegyzetek/blob/master/Algopro/NLP%20Embedding%20student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import re
from collections import Counter
from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report

c:\Users\Balint\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
VOCAB_SIZE = 20000
EMBEDDING_DIM = 100
MIN_COUNT = 5

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load QQP
qqp = load_dataset("glue", "qqp")
train_df = qqp["train"].to_pandas().dropna(subset=["question1", "question2"]).reset_index(drop=True)
val_df = qqp["validation"].to_pandas().dropna(subset=["question1", "question2"]).reset_index(drop=True)
print(f"Train: {len(train_df)}, Val: {len(val_df)}")

Train: 363846, Val: 40430


In [12]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    return text.strip().split()

In [ ]:
# create vocabulary
word_counts = Counter([tok for col in ["question1", "question2"] for x in train_df[col] for tok in preprocess(x)])
idx2word = [key for key in word_counts.keys() if word_counts[key] >= MIN_COUNT]
idx2word.insert(0, '<PAD>')
idx2word.insert(1, '<UNK>')
idx2word = idx2word[:VOCAB_SIZE]
word2idx = {idx2word[i]:i for i in range(len(idx2word))}

In [26]:
def text_to_indices(text, word2idx):
    prep = preprocess(text)
    return [word2idx.get(w, 1) for w in prep]

In [30]:
class QQPDataset(Dataset):
    def __init__(self, df, word2idx):
        self.labels = df["label"].tolist()
        self.q1 = [text_to_indices(t, word2idx) for t in df["question1"]]
        self.q2 = [text_to_indices(t, word2idx) for t in df["question2"]]
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return (
            torch.tensor(self.q1[idx], dtype=torch.long),
            torch.tensor(self.q2[idx], dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.float),
        )

In [31]:
def collate_fn(batch):
    q1s, q2s, labels = zip(*batch)
    q1s = nn.utils.rnn.pad_sequence(q1s, batch_first=True, padding_value=0)
    q2s = nn.utils.rnn.pad_sequence(q2s, batch_first=True, padding_value=0)
    labels = torch.stack(labels)
    return q1s, q2s, labels

train_loader = DataLoader(QQPDataset(train_df, word2idx), batch_size=256, collate_fn=collate_fn, shuffle=True)
val_loader = DataLoader(QQPDataset(val_df, word2idx), batch_size=256, collate_fn=collate_fn)

In [40]:
x = torch.ones(10) *5
x[:5] = 0
mask = torch.where(x != 0, 1, 0)
mask

tensor([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])

In [75]:
class QQPModel(nn.Module):
    def __init__(self, num_words, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(num_words, embed_dim, padding_idx=0)
        # Input: concat of q1, q2, q1*q2 = 3 * embed_dim
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim*3, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

    def encode(self, x):
        mask = torch.where(x != 0, 1, 0)
        counts = mask.sum(dim=1).clamp(1e-9).unsqueeze(1)
        emb = self.embedding(x)
        emb = (emb * mask.unsqueeze(-1)).sum(dim=1)
        return emb / counts

    def forward(self, q1, q2):
        q1_emb = self.encode(q1)
        q2_emb = self.encode(q2)
        return self.classifier(torch.cat([q1_emb, q2_emb, q1_emb*q2_emb], dim=-1)).squeeze(1)

In [78]:
from tqdm import tqdm

model = QQPModel(VOCAB_SIZE, EMBEDDING_DIM).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), 0.0001)

for epoch in range(10):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for q1, q2, labels in tqdm(train_loader):
        q1, q2, labels = q1.to(device), q2.to(device), labels.to(device)
        preds = model(q1, q2)
        loss = criterion(preds, labels)
        total_loss += loss.item()
        correct += (labels == (preds > 0)).sum().item()
        total += (labels == labels).sum().item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    model.eval()
    v_correct = v_total = 0
    with torch.no_grad():
        for q1, q2, labels in val_loader:
            q1, q2, labels = q1.to(device), q2.to(device), labels.to(device)
            out = model(q1, q2)
            v_correct += ((out > 0).float() == labels).sum().item()
            v_total += len(labels)

    print(f"Epoch {epoch+1}/10 | Loss: {total_loss/total:.4f} | Train: {correct/total:.4f} | Val: {v_correct/v_total:.4f}")

100%|██████████| 1422/1422 [00:08<00:00, 169.25it/s]


Epoch 1/10 | Loss: 0.0021 | Train: 0.7140 | Val: 0.7397


100%|██████████| 1422/1422 [00:08<00:00, 172.67it/s]


Epoch 2/10 | Loss: 0.0019 | Train: 0.7528 | Val: 0.7544


100%|██████████| 1422/1422 [00:08<00:00, 166.69it/s]


...

Epoch 8/10 | Loss: 0.0015 | Train: 0.8097 | Val: 0.7807


100%|██████████| 1422/1422 [00:09<00:00, 148.97it/s]


Epoch 9/10 | Loss: 0.0015 | Train: 0.8150 | Val: 0.7798


100%|██████████| 1422/1422 [00:08<00:00, 158.87it/s]


Epoch 10/10 | Loss: 0.0015 | Train: 0.8206 | Val: 0.7827


In [79]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for q1, q2, labels in val_loader:
        q1, q2, labels = q1.to(device), q2.to(device), labels.to(device)
        out = model(q1, q2)
        preds = (out > 0).float()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(classification_report(
    all_labels, all_preds,
    target_names=["Not Duplicate", "Duplicate"],
    digits=4
))

               precision    recall  f1-score   support

Not Duplicate     0.8403    0.8101    0.8249     25545
    Duplicate     0.6931    0.7357    0.7137     14885

     accuracy                         0.7827     40430
    macro avg     0.7667    0.7729    0.7693     40430
 weighted avg     0.7861    0.7827    0.7840     40430

